# Klasifikasi Tiket IT Helpdesk - Hybrid SVM-GenAI

Notebook ini mendemonstrasikan **cara menjalankan pipeline komparasi** (SVM, RF, LR, BERT, Hybrid Fusion, Hybrid Voting) yang ada di [`src/compare_svm_genai.py`](../src/compare_svm_genai.py).

## Skema yang Dibandingkan

| # | Skema | Pendekatan |
|---|-------|------------|
| 1 | SVM | TF-IDF + LinearSVC |
| 2 | Random Forest | TF-IDF + RandomForest |
| 3 | Logistic Regression | TF-IDF + LogisticRegression |
| 4 | BERT | Fine-tuned DistilBERT (opsional, lambat di CPU) |
| 5 | **Hybrid SVM-GenAI (Fusion)** | **TF-IDF + OpenAI Embedding -> LinearSVC** (winning variant) |
| 6 | Hybrid SVM-GenAI (Voting) | Majority vote: SVM + Fusion + LLM (opsional, mahal) |

## Hasil 5-Fold CV (filtered dataset, 19 kategori)

| Model | Acc Cat | F1 Cat | Acc Pri | F1 Pri |
|---|---|---|---|---|
| **Hybrid Fusion** (winner) | **0.8214 +/- 0.003** | **0.6738 +/- 0.009** | **0.7219 +/- 0.009** | **0.7097 +/- 0.008** |
| SVM | 0.8125 +/- 0.004 | 0.6535 +/- 0.015 | 0.7167 +/- 0.011 | 0.7033 +/- 0.009 |
| Random Forest | 0.7660 +/- 0.005 | 0.5353 +/- 0.018 | 0.7172 +/- 0.009 | 0.6838 +/- 0.010 |
| Logistic Regression | 0.7764 +/- 0.005 | 0.4713 +/- 0.007 | 0.6335 +/- 0.008 | 0.5864 +/- 0.008 |


## 1. Setup

Pastikan dependencies sudah terinstall dan `.env` berisi `OPENAI_API_KEY`.

In [ ]:
# Uncomment kalau belum install:
# !pip install -r ../requirements.txt

import sys
from pathlib import Path

# Tambahkan src/ ke path agar bisa import compare_svm_genai
sys.path.insert(0, str(Path('../src').resolve()))

import pandas as pd
from compare_svm_genai import run_pipeline, run_pipeline_kfold

## 2. Run Pipeline (Single 80/20 Split)

Train semua model di train set, evaluasi di test set. **Pakai filtered dataset (19 kategori)** - hasil terbaik. Skip BERT karena sangat lambat di CPU.

In [ ]:
run_pipeline(
    input_file='../data/cobacek_filtered.xlsx',
    output_file='../results/cobacek_filtered_compare.xlsx',
    category_col='category_filtered',  # 19 kelas (vs 81 di kolom 'category')
    skip_bert=True,                     # BERT sangat lambat di CPU
    enable_voting=False,                # Voting opsional, mahal
    random_state=42,
)

## 3. Baca & Tampilkan Hasil

In [ ]:
metrics = pd.read_excel('../results/cobacek_filtered_compare.xlsx', sheet_name='Metrics')
metrics_display = metrics.copy()
for col in ['accuracy', 'macro_precision', 'macro_recall', 'macro_f1', 'weighted_f1']:
    metrics_display[col] = metrics_display[col].round(4)
metrics_display

## 4. Visualisasi Heatmap

In [ ]:
from visualize_results import load_metrics, pivot_for_heatmap, plot_heatmap

metrics = load_metrics('../results/cobacek_filtered_compare.xlsx')
table = pivot_for_heatmap(metrics)
plot_heatmap(table, '../results/heatmap_filtered.png',
             title='PERBANDINGAN HYBRID SVM-GENAI (FILTERED, 19 KATEGORI)')

from IPython.display import Image
Image('../results/heatmap_filtered.png')

## 5. (Opsional) 5-Fold Cross Validation

Untuk paper credibility - angka mean +/- std lintas 5 fold. Estimasi waktu ~50 menit, biaya ~$0.15.

In [ ]:
# Uncomment untuk run (lama!):
# run_pipeline_kfold(
#     input_file='../data/cobacek_filtered.xlsx',
#     output_file='../results/cobacek_filtered_kfold.xlsx',
#     n_folds=5,
#     base_seed=42,
#     category_col='category_filtered',
#     skip_bert=True,
# )

# Lihat hasil 5-fold yang sudah di-run
agg = pd.read_excel('../results/cobacek_filtered_kfold.xlsx', sheet_name='Metrics_Aggregated')
agg_display = agg.copy()
for col in agg_display.select_dtypes(include='float').columns:
    agg_display[col] = agg_display[col].round(4)
agg_display

## 6. (Opsional) Hybrid Voting Ensemble

Kombinasi 3 voter: SVM + Hybrid Fusion + LLM (gpt-4.1-mini, prediksi independen). Tie-break: Fusion.

**Sangat mahal**: 1 GenAI call per test row -> untuk 3.268 baris ~$2-3, ~3 jam.

In [ ]:
# Uncomment untuk run:
# run_pipeline(
#     input_file='../data/cobacek_filtered.xlsx',
#     output_file='../results/cobacek_filtered_voting.xlsx',
#     category_col='category_filtered',
#     skip_bert=True,
#     enable_voting=True,
#     forced_model='gpt-4.1-mini',
#     random_state=42,
# )

## CLI Alternative

Semua di atas bisa dijalankan via command line dari project root:

```bash
# Single split
python src/compare_svm_genai.py \
    --input data/cobacek_filtered.xlsx \
    --category-col category_filtered \
    --skip-bert \
    --output results/cobacek_filtered_compare.xlsx

# 5-fold CV
python src/compare_svm_genai.py \
    --input data/cobacek_filtered.xlsx \
    --category-col category_filtered \
    --skip-bert --n-folds 5 \
    --output results/cobacek_filtered_kfold.xlsx

# Voting ensemble
python src/compare_svm_genai.py \
    --input data/cobacek_filtered.xlsx \
    --category-col category_filtered \
    --skip-bert --enable-voting \
    --model gpt-4.1-mini \
    --output results/cobacek_filtered_voting.xlsx
```